# NOFE Cognitive Classification — Low / High Performance Groups

Instead of predicting exact scores (regression), discretise cognitive scores into
**2 groups** (median split: low / high) and compare classifiers using LOO-CV.

Classification is more robust than regression at n=16 and directly answers:
> *Can white-matter microstructure (NOFE) distinguish high vs low cognitive performers?*

| Feature | Classifiers |
|---|---|
| NOFE-Global (32d) | SVM-RBF, Random Forest, KNN |
| NOFE-WM+Region (160d) | SVM-RBF, Random Forest, KNN |
| PCA-FullBrain (15d) | SVM-RBF, Random Forest, KNN |

In [1]:
import sys, json, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                              confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

project_root = Path('../../').resolve()
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root / 'dMRI' / 'scripts'))

from utilities import KernelNN
from utilities_dMRI import dMRI_Graph_Generator

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

using Device: mps
Device: mps


## 1. Load NOFE Model

In [2]:
model_type   = 'regular'
model_prefix = 'base'

trained_models_path = project_root / 'dMRI' / 'trained_models' / model_type
model_id = sorted([f.stem for f in trained_models_path.glob(f'{model_prefix}_*.pth')])[-1]
print(f'Model: {model_id}')

with open(trained_models_path / 'logs' / 'configs' / f'{model_id}.json') as f:
    config = json.load(f)

model = KernelNN(
    width=config['width'], ker_width=config['ker_width'], depth=config['depth'],
    in_width=config['node_features'], ker_in=config['edge_attr'], out_width=config['out_width'],
)
model.load_state_dict(torch.load(trained_models_path / f'{model_id}.pth', map_location='cpu'))
model.eval().to(device)
EMBED_DIM = config['width']
print(f'Embedding dim: {EMBED_DIM}')

@torch.no_grad()
def extract_nofe(graph):
    x = model.fc1(graph.x.to(device))
    for _ in range(model.depth):
        x = F.relu(model.conv1(x, graph.edge_index.to(device), graph.edge_attr.to(device)))
    return x.cpu().numpy()

REGION_NAMES = ['frontal', 'parietal', 'temporal', 'occipital', 'subcortical']

def region_masks(coords):
    y, z = coords[:, 1], coords[:, 2]
    return {
        'frontal':     y > 0.60,
        'parietal':    (y > 0.35) & (y < 0.85) & (z > 0.62),
        'temporal':    (y > 0.20) & (y < 0.70) & (z < 0.55),
        'occipital':   y < 0.25,
        'subcortical': (y > 0.35) & (y < 0.65) & (z > 0.28) & (z < 0.58),
    }

def region_mean(hidden, coords):
    masks = region_masks(coords)
    return np.concatenate([
        hidden[m].mean(axis=0) if m.sum() > 0 else np.zeros(EMBED_DIM)
        for m in masks.values()
    ])

Model: base_2026-04-29_t11-16-23
Embedding dim: 32


## 2. Extract / Cache Subject Features

In [3]:
from dipy.reconst.dti import TensorModel
from dipy.core.gradients import gradient_table as dipy_gtab

CACHE_DIR  = Path('../cache')
CACHE_DIR.mkdir(exist_ok=True)
CACHE_FILE = CACHE_DIR / 'nofe_subject_features_full.npz'

data_root    = project_root / 'dMRI' / 'data'
DIFF_SUBPATH = 'T1w/Diffusion_7T/'
N_VOXELS     = 6000   # 降低中间张量 (6000,10,143)=34MB vs 57MB
N_REPEATS    = 6      # 降低图构建次数
FA_THRESHOLD = 0.2

def _flush_mps():
    if device.type == 'mps':
        torch.mps.empty_cache()

local_subjects = sorted([d for d in data_root.iterdir() if d.is_dir() and d.name.isdigit()])

if CACHE_FILE.exists():
    cache = np.load(CACHE_FILE)
    X_nofe        = cache['X_nofe']
    X_nofe_wm_reg = cache['X_nofe_wm_reg']
    X_raw         = cache['X_raw']
    subject_ids   = cache['subject_ids'].tolist()
    print(f'Loaded from cache — {len(subject_ids)} subjects')
    print(f'  X_nofe        : {X_nofe.shape}')
    print(f'  X_nofe_wm_reg : {X_nofe_wm_reg.shape}')
else:
    Generator = dMRI_Graph_Generator(str(data_root))
    global_l, wm_reg_l, raw_l, subject_ids = [], [], [], []

    for subj_dir in local_subjects:
        path  = str(subj_dir / DIFF_SUBPATH) + '/'
        bvals = np.loadtxt(path + 'bvals')
        bvecs = np.loadtxt(path + 'bvecs')
        print(f'  {subj_dir.name} — FA ...', end=' ', flush=True)

        Generator.load_data(path=path)
        orig_da     = Generator.da
        orig_coords = Generator.normed_coords

        # DTI 白质掩膜
        b1000_sel = bvals < 1200
        gtab = dipy_gtab(bvals[b1000_sel], bvecs=bvecs[:, b1000_sel], b0_threshold=100)
        data_dti = orig_da.detach().cpu().numpy()[:, b1000_sel]
        fa = np.clip(TensorModel(gtab).fit(data_dti).fa, 0, 1)
        del data_dti                              # 释放中间数组
        wm_bool = torch.from_numpy(fa > FA_THRESHOLD)
        del fa

        da_np = orig_da.detach().cpu().numpy()
        raw_l.append(da_np.mean(axis=0))
        del da_np

        print(f'WM={wm_bool.float().mean()*100:.0f}% — features ...', end=' ', flush=True)

        # 全脑采样
        Generator.da = orig_da; Generator.normed_coords = orig_coords
        h_list, c_list = [], []
        for _ in range(N_REPEATS):
            g = Generator.sample_graph(
                N=N_VOXELS, k=config['k'], perplexity=config['perplexity'],
                sampling_method='random', gen_feature_mode=config['feature_mode'],
                sigma_method=config['sigma_method'])
            h = extract_nofe(g); c = g.normed_coords.cpu().numpy()
            h_list.append(h); c_list.append(c)
            del g; _flush_mps()
        global_l.append(np.stack([h.mean(axis=0) for h in h_list]).mean(axis=0))
        del h_list, c_list

        # 白质采样
        Generator.da = orig_da[wm_bool]; Generator.normed_coords = orig_coords[wm_bool]
        wh_list, wc_list = [], []
        for _ in range(N_REPEATS):
            g = Generator.sample_graph(
                N=N_VOXELS, k=config['k'], perplexity=config['perplexity'],
                sampling_method='random', gen_feature_mode=config['feature_mode'],
                sigma_method=config['sigma_method'])
            wh_list.append(extract_nofe(g)); wc_list.append(g.normed_coords.cpu().numpy())
            del g; _flush_mps()
        wm_reg_l.append(np.stack([region_mean(h, c)
                                   for h, c in zip(wh_list, wc_list)]).mean(axis=0))
        del wh_list, wc_list, orig_da, orig_coords, wm_bool

        subject_ids.append(int(subj_dir.name))
        print('done')
        gc.collect(); _flush_mps()   # 每个被试结束后清空 MPS heap

    X_nofe        = np.stack(global_l)
    X_nofe_wm_reg = np.stack(wm_reg_l)
    X_raw         = np.stack(raw_l)
    np.savez(CACHE_FILE, X_nofe=X_nofe, X_nofe_wm_reg=X_nofe_wm_reg,
             X_raw=X_raw, subject_ids=np.array(subject_ids))
    print(f'\nSaved to cache: {CACHE_FILE}')

# PCA
n_pca = min(EMBED_DIM, len(X_raw) - 1)
X_pca = PCA(n_components=n_pca, random_state=0).fit_transform(
    StandardScaler().fit_transform(X_raw))

FEATURE_SETS = {
    'NOFE-Global':    X_nofe,
    'NOFE-WM+Region': X_nofe_wm_reg,
    'PCA-FullBrain':  X_pca,
}
for k, v in FEATURE_SETS.items():
    print(f'  {k:20s}: {v.shape}')

Loaded from cache — 16 subjects
  X_nofe        : (16, 32)
  X_nofe_wm_reg : (16, 160)
  NOFE-Global         : (16, 32)
  NOFE-WM+Region      : (16, 160)
  PCA-FullBrain       : (16, 15)


## 3. Load Cognitive Scores + Discretise

In [ ]:
csv_path = sorted(data_root.glob('HCP_YA_subjects_*.csv'))[-1]
df = pd.read_csv(csv_path)

TARGETS = {
    'CardSort':  'CardSort_Unadj',
    'Flanker':   'Flanker_Unadj',
    'PMAT24':    'PMAT24_A_CR',
    'ProcSpeed': 'ProcSpeed_Unadj',
    'ListSort':  'ListSort_Unadj',
    'ReadEng':   'ReadEng_Unadj',
    'PicVocab':  'PicVocab_Unadj',
}
TARGET_NAMES = list(TARGETS.keys())

local_df = df[df['Subject'].isin(subject_ids)].set_index('Subject')
Y_cont = np.stack([local_df.loc[subject_ids, col].values.astype(float)
                   for col in TARGETS.values()], axis=1)  # (N, 7) continuous

# Binary classification: median split (0=low, 1=high)
medians   = np.median(Y_cont, axis=0)
Y_binary  = (Y_cont >= medians).astype(int)  # (N, 7)

print('Binary label distribution (median split):')
for i, name in enumerate(TARGET_NAMES):
    n_high = Y_binary[:, i].sum()
    print(f'  {name:12s}: low={len(subject_ids)-n_high}  high={n_high}  '
          f'median={medians[i]:.1f}')

Binary label distribution (median split):
  CardSort    : low=8  high=8  median=114.7
  Flanker     : low=8  high=8  median=106.9
  PMAT24      : low=8  high=8  median=14.5
  ProcSpeed   : low=8  high=8  median=115.3
  ListSort    : low=8  high=8  median=106.1
  ReadEng     : low=8  high=8  median=119.8
  PicVocab    : low=8  high=8  median=115.2


: 

## 4. LOO-CV Classification

In [ ]:
CLASSIFIERS = {
    'SVM-RBF':    lambda: SVC(kernel='rbf', C=1.0, random_state=0),
    'RandForest': lambda: RandomForestClassifier(n_estimators=100, max_features='sqrt',
                                                  random_state=0, n_jobs=1),
    'KNN-3':      lambda: KNeighborsClassifier(n_neighbors=3),
}

results = {}  # (feat_name, clf_name, score_name) -> (acc, bacc, cm)

for fname, X in FEATURE_SETS.items():
    for cname, clf_fn in CLASSIFIERS.items():
        key = (fname, cname)
        results[key] = {}
        for i, sname in enumerate(TARGET_NAMES):
            y = Y_binary[:, i]
            loo = LeaveOneOut()
            true_all, pred_all = [], []
            for train_idx, test_idx in loo.split(X):
                clf = clf_fn()   # 每折重新实例化，避免状态污染
                sx  = StandardScaler().fit(X[train_idx])
                clf.fit(sx.transform(X[train_idx]), y[train_idx])
                pred_all.append(int(clf.predict(sx.transform(X[test_idx]))[0]))
                true_all.append(int(y[test_idx][0]))
            yt, yp = np.array(true_all), np.array(pred_all)
            acc  = accuracy_score(yt, yp)
            bacc = balanced_accuracy_score(yt, yp)
            # 保证 cm 始终是 2×2
            cm   = confusion_matrix(yt, yp, labels=[0, 1])
            results[key][sname] = (acc, bacc, cm)

        mean_bacc = np.mean([results[key][s][1] for s in TARGET_NAMES])
        print(f'{fname:20s} | {cname:12s}: mean balanced-acc = {mean_bacc:.3f}')
    print()

NOFE-Global          | SVM-RBF     : mean balanced-acc = 0.277


## 5. Accuracy Heatmap

In [ ]:
n_feat = len(FEATURE_SETS)
n_clf  = len(CLASSIFIERS)
n_score = len(TARGET_NAMES)

# Build balanced-accuracy matrix: rows = (feat × clf), cols = score
row_labels = [f'{f}\n{c}' for f in FEATURE_SETS for c in CLASSIFIERS]
acc_mat    = np.zeros((n_feat * n_clf, n_score))
for ri, (fname, cname) in enumerate([(f, c) for f in FEATURE_SETS for c in CLASSIFIERS]):
    for ci, sname in enumerate(TARGET_NAMES):
        acc_mat[ri, ci] = results[(fname, cname)][sname][1]  # balanced acc

fig, ax = plt.subplots(figsize=(12, max(4, n_feat * n_clf * 0.7 + 1)), facecolor='white')
im = ax.imshow(acc_mat, aspect='auto', cmap='RdYlGn', vmin=0.3, vmax=1.0)

ax.set_xticks(range(n_score))
ax.set_xticklabels(TARGET_NAMES, rotation=30, ha='right', fontsize=11)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=8)

# Horizontal separator between feature sets
for k in range(1, n_feat):
    ax.axhline(k * n_clf - 0.5, color='white', lw=2)

for i in range(len(row_labels)):
    for j in range(n_score):
        v = acc_mat[i, j]
        star = ' ★' if acc_mat[:, j].argmax() == i else ''
        ax.text(j, i, f'{v:.2f}{star}', ha='center', va='center', fontsize=8.5,
                color='white' if v < 0.45 else '#222', fontweight='bold')

plt.colorbar(im, ax=ax, label='Balanced accuracy', fraction=0.015)
ax.set_title('LOO-CV Balanced Accuracy  |  ★ = best per score',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Chance level reminder
print('Chance level (random): 0.50  |  Balanced accuracy accounts for class imbalance.')

## 6. Confusion Matrices for Best Method per Score

In [ ]:
ncols = 4
nrows = (n_score + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.5),
                          facecolor='white')
axes = np.array(axes).reshape(-1)

for i, sname in enumerate(TARGET_NAMES):
    best_row = acc_mat[:, i].argmax()
    best_feat, best_clf = [(f, c) for f in FEATURE_SETS for c in CLASSIFIERS][best_row]
    _, bacc, cm = results[(best_feat, best_clf)][sname]
    ax = axes[i]
    ConfusionMatrixDisplay(cm, display_labels=['Low', 'High']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{sname}\n{best_feat.split("-")[0]} | {best_clf}\nBAcc={bacc:.2f}',
                 fontsize=8.5)

for ax in axes[n_score:]:
    ax.axis('off')

fig.suptitle('Confusion matrices — best method per cognitive score (LOO-CV)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Random Forest Feature Importance

In [ ]:
# Train RF on full dataset for each score (not LOO) to inspect importance
FEAT_FOR_IMP = 'NOFE-WM+Region'
X_imp = FEATURE_SETS[FEAT_FOR_IMP]
n_dims = X_imp.shape[1]  # 160

# Feature name: region × embedding_dim
reg_labels = [f'{r[:3].upper()}_{d}' for r in REGION_NAMES for d in range(EMBED_DIM)]

fig, axes = plt.subplots(1, len(TARGET_NAMES), figsize=(4 * len(TARGET_NAMES), 4),
                          facecolor='white')

for ax, sname in zip(axes, TARGET_NAMES):
    y = Y_binary[:, TARGET_NAMES.index(sname)]
    sx = StandardScaler().fit(X_imp)
    rf = RandomForestClassifier(n_estimators=500, random_state=0)
    rf.fit(sx.transform(X_imp), y)
    imp = rf.feature_importances_

    # Aggregate by region (sum per region)
    reg_imp = np.stack([imp[k*EMBED_DIM:(k+1)*EMBED_DIM].sum()
                        for k in range(len(REGION_NAMES))])
    reg_imp /= reg_imp.sum()

    ax.barh(REGION_NAMES, reg_imp, color='steelblue', alpha=0.85)
    ax.set_xlabel('Importance', fontsize=8)
    ax.set_title(sname, fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=7)
    ax.set_xlim(0, reg_imp.max() * 1.25)
    for j, v in enumerate(reg_imp):
        ax.text(v + 0.005, j, f'{v:.2f}', va='center', fontsize=7)

fig.suptitle(f'RF Feature Importance by Region ({FEAT_FOR_IMP}, trained on full dataset)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Summary Table

In [ ]:
rows = []
for fname in FEATURE_SETS:
    for cname in CLASSIFIERS:
        for sname in TARGET_NAMES:
            acc, bacc, cm = results[(fname, cname)][sname]
            rows.append({
                'Feature':   fname,
                'Classifier': cname,
                'Score':     sname,
                'Accuracy':  round(acc, 3),
                'Balanced Acc': round(bacc, 3),
            })

df_res = pd.DataFrame(rows)

# Pivot: mean balanced accuracy per (feature, classifier)
pivot = df_res.groupby(['Feature', 'Classifier'])['Balanced Acc'].mean().round(3).unstack()
print('Mean balanced accuracy per (Feature, Classifier):')
print(pivot.to_string())

# Best overall
best_row = df_res.groupby(['Feature', 'Classifier'])['Balanced Acc'].mean().idxmax()
print(f'\nBest combination: Feature={best_row[0]}  Classifier={best_row[1]}')
df_res